# 04 - Embeddings y similitud

Los embeddings convierten texto en vectores. Eso permite comparar significado con operaciones numéricas.


## Paso 0: imports y funciones base

La similitud coseno compara la dirección de dos vectores. Valores más altos indican mayor parecido semántico.


In [ ]:
import numpy as np
import ollama

MODEL = "llama3.2"

def obtener_embedding(texto):
    return ollama.embeddings(model=MODEL, prompt=texto)["embedding"]

def similitud_coseno(v1, v2):
    return np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))


## Paso 1: ver un embedding

El vector completo suele tener muchas dimensiones. No se interpreta número por número; se usa para comparar textos.


In [ ]:
vector = obtener_embedding("Hola mundo")
print(f"Dimensiones: {len(vector)}")
print(f"Primeros 5 valores: {vector[:5]}")


## Paso 2: comparar frases

Aquí creamos embeddings para frases parecidas y no parecidas, y luego calculamos una matriz de similitud.


In [ ]:
frases = [
    "El perro juega en el parque",
    "Un canino se divierte al aire libre",
    "La bolsa de valores subió un 5%",
    "El gato duerme en el sofá",
]

embeddings = [obtener_embedding(frase) for frase in frases]

print(f"{'':>35}", end="")
for indice in range(len(frases)):
    print(f"  F{indice + 1}", end="")
print()

for i, frase in enumerate(frases):
    print(f"F{i + 1}: {frase[:30]:>30}  ", end="")
    for embedding in embeddings:
        print(f"{similitud_coseno(embeddings[i], embedding):.2f}", end="  ")
    print()


## Paso 3: buscador semántico

En vez de buscar palabras exactas, convertimos la pregunta a embedding y recuperamos los documentos más cercanos.


In [ ]:
base = [
    "Python fue creado por Guido van Rossum en 1991",
    "JavaScript fue creado por Brendan Eich en 1995",
    "Los arrays en Python se llaman listas",
    "El machine learning es una rama de la inteligencia artificial",
    "Git fue creado por Linus Torvalds en 2005",
    "Las redes neuronales se inspiran en el cerebro humano",
]

embeddings_base = [obtener_embedding(texto) for texto in base]

def buscar(consulta, top_k=3):
    embedding_consulta = obtener_embedding(consulta)
    resultados = []
    for texto, embedding in zip(base, embeddings_base):
        similitud = similitud_coseno(embedding_consulta, embedding)
        resultados.append((similitud, texto))
    return sorted(resultados, reverse=True)[:top_k]


## Paso 4: probar búsquedas

Observa que la búsqueda puede encontrar documentos relevantes aunque la pregunta no use exactamente las mismas palabras.


In [ ]:
for consulta in ["¿quién inventó un lenguaje de programación?", "¿cómo funciona la IA?"]:
    print(consulta)
    for similitud, texto in buscar(consulta):
        print(f"[{similitud:.3f}] {texto}")
    print()
